# Приведение названий филиалов в `_test` к старому виду

DAG за август–ноябрь записал сырые названия из ЦФТ («АМУРСКИЙ РФ АО "РОССЕЛ…"»), а в истории и в таблице прав `rls_acq_user_filial` они в виде «Амурский РФ». Тетрадка переписывает только колонку `filial_rf`, только в `sbx_da.tmp_shestopalov_acq_fin_version_test`, только месяцы с `2026-08`. Таблицу дашборда без `_test` не трогает.

Запускать сверху вниз под своей учёткой DRP. Если LDAP отказал, не повторяйте больше 2 раз, чтобы не заблокировать учётку.

In [ ]:
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 250)

DRP_TEST = "sbx_da.tmp_shestopalov_acq_fin_version_test"
FROM_MONTH = "2026-08"
DRP_USER_DEFAULT = "Shestopalov-VYur"

assert DRP_TEST.endswith("_test")

if "drp" not in globals() or drp is None:
    drp_user = input("DRP user: ").strip() or DRP_USER_DEFAULT
    drp = connect(to="DRP", user_params={"user_name": drp_user, "password": getpass("DRP password: ")})


def fetch(sql):
    with drp:
        df = drp.fetch(sql)
    return pd.DataFrame() if df is None else df


def execute(sql):
    with drp:
        drp.execute(sql)


display(fetch("SELECT current_user AS me"))

## 1) Правило нормализации и проверка до изменения

То же правило, что в `load_acq_temp2target.sql` DAG. Норма: `rows_to_change` больше 0, по месяцам с августа `filials` заметно больше, чем в июле (~63).

In [ ]:
NORM_MAP_SQL = rf"""
SELECT raw,
       REGEXP_REPLACE(UPPER(LEFT(s2, 1)) || LOWER(SUBSTR(s2, 2)), 'рф', 'РФ', 'gi') AS norm
FROM (
    SELECT raw,
           NULLIF(BTRIM(REGEXP_REPLACE(s1, '\m(ао|пао|оао|зао)\M.*$', '', 'i'), ' ,;.-'), '') AS s2
    FROM (
        SELECT raw,
               CASE WHEN STRPOS(LOWER(s0), 'рф') > 0 THEN LEFT(s0, STRPOS(LOWER(s0), 'рф') + 1) ELSE s0 END AS s1
        FROM (
            SELECT raw,
                   NULLIF(BTRIM(REGEXP_REPLACE(REPLACE(CAST(raw AS TEXT), CHR(160), ' '), '\s+', ' ', 'g')), '') AS s0
            FROM (
                SELECT DISTINCT filial_rf AS raw
                FROM {DRP_TEST}
                WHERE report_month >= '{FROM_MONTH}'
                  AND filial_rf IS NOT NULL
            ) r
        ) a
    ) b
) c
"""

BY_MONTH_SQL = f"""
SELECT SUBSTRING(BTRIM(CAST(report_month AS TEXT)) FROM 1 FOR 7) AS m,
       COUNT(*) AS rows_cnt,
       COUNT(DISTINCT filial_rf) AS filials
FROM {DRP_TEST}
GROUP BY 1
ORDER BY 1
"""

TO_CHANGE_SQL = f"""
SELECT COUNT(*) AS rows_to_change
FROM {DRP_TEST} d
JOIN ({NORM_MAP_SQL}) n ON d.filial_rf = n.raw
WHERE d.report_month >= '{FROM_MONTH}'
  AND d.filial_rf IS DISTINCT FROM n.norm
"""

before = fetch(BY_MONTH_SQL)
display(before)
to_change = int(fetch(TO_CHANGE_SQL).iloc[0, 0])
print("Строк к изменению:", to_change)

## 2) Обновление `filial_rf` в `_test`

Меняет только `filial_rf`, остальные колонки и строки не трогает. Повторный запуск безопасен: изменять будет нечего.

In [ ]:
UPDATE_SQL = f"""
UPDATE {DRP_TEST} AS d
SET filial_rf = n.norm
FROM ({NORM_MAP_SQL}) n
WHERE d.report_month >= '{FROM_MONTH}'
  AND d.filial_rf = n.raw
  AND d.filial_rf IS DISTINCT FROM n.norm
"""

if to_change == 0:
    print("Менять нечего")
else:
    execute(UPDATE_SQL)
    print("UPDATE выполнен")

## 3) Проверка после

Норма: `rows_cnt` по месяцам не изменился, `filials` с августа примерно как в июле (~63), строк к изменению 0, в списке нет названий заглавными буквами и с «АО».

In [ ]:
after = fetch(BY_MONTH_SQL)
display(before.merge(after, on="m", suffixes=("_before", "_after")))
print("Строк к изменению осталось:", int(fetch(TO_CHANGE_SQL).iloc[0, 0]))

display(fetch(f"""
SELECT COALESCE(filial_rf, '<NULL>') AS filial_rf, COUNT(*) AS rows_cnt
FROM {DRP_TEST}
WHERE report_month >= '{FROM_MONTH}'
GROUP BY 1
ORDER BY 1
"""))

## 4) Пересчёт `tariff_short` в `_test` с июля

DAG считал `tariff_short` в Impala, где `lower()` не меняет кириллицу, и почти всё попало в «Не сегментирован». Правило то же, что в старом скрипте и в `load_acq_temp2target.sql`. Меняется только `tariff_short`, только в `_test`, только с `2026-07`.

Норма до: с июля почти всё «Не сегментирован». Норма после: распределение как в июне (Акционный ~2,4–2,5 тыс., Индивидуальный ~2,6–2,7 тыс., Стандарт ~1,6 тыс., По актам ~30, Не сегментирован единицы).

In [ ]:
TARIFF_FROM_MONTH = "2026-07"

TARIFF_RULE = """CASE
  WHEN STRPOS(LOWER(COALESCE(CAST(tariff_name AS text), '')), 'акт') > 0 THEN 'По актам'
  WHEN STRPOS(LOWER(COALESCE(CAST(tariff_name AS text), '')), 'акцион') > 0
    OR STRPOS(LOWER(COALESCE(CAST(tariff_name AS text), '')), 'меню') > 0
    OR STRPOS(LOWER(COALESCE(CAST(tariff_name AS text), '')), 'сезон') > 0 THEN 'Акционный'
  WHEN STRPOS(LOWER(COALESCE(CAST(tariff_name AS text), '')), 'индив') > 0 THEN 'Индивидуальный'
  WHEN STRPOS(LOWER(COALESCE(CAST(tariff_name AS text), '')), 'стандарт') > 0 THEN 'Стандарт'
  ELSE 'Не сегментирован'
END"""

TARIFF_BY_MONTH_SQL = f"""
SELECT SUBSTRING(BTRIM(CAST(report_month AS TEXT)) FROM 1 FOR 7) AS m,
       tariff_short,
       COUNT(*) AS rows_cnt
FROM {DRP_TEST}
WHERE report_month >= '2026-06'
GROUP BY 1, 2
ORDER BY 1, 2
"""

TARIFF_TO_CHANGE_SQL = f"""
SELECT COUNT(*) AS rows_to_change
FROM {DRP_TEST}
WHERE report_month >= '{TARIFF_FROM_MONTH}'
  AND tariff_short IS DISTINCT FROM {TARIFF_RULE}
"""

tariff_before = fetch(TARIFF_BY_MONTH_SQL)
tariff_to_change = int(fetch(TARIFF_TO_CHANGE_SQL).iloc[0, 0])
print("Строк к изменению:", tariff_to_change)

if tariff_to_change == 0:
    print("Менять нечего")
else:
    execute(f"""
    UPDATE {DRP_TEST}
    SET tariff_short = {TARIFF_RULE}
    WHERE report_month >= '{TARIFF_FROM_MONTH}'
      AND tariff_short IS DISTINCT FROM {TARIFF_RULE}
    """)
    print("UPDATE выполнен")

print("Строк к изменению осталось:", int(fetch(TARIFF_TO_CHANGE_SQL).iloc[0, 0]))
tariff_after = fetch(TARIFF_BY_MONTH_SQL)
display(
    tariff_before.merge(tariff_after, on=["m", "tariff_short"], how="outer", suffixes=("_before", "_after"))
    .fillna(0)
    .sort_values(["m", "tariff_short"])
)